# 13.3. 지표와 지표표

In [1]:
import numpy as np

D = np.array([[1, 0, -6, 0], [0, 2, -9, 0], [5, 0, 3, 7], [-1, 5, 0, 4]])

T = np.array([[1, 2, 3, 4], [np.sqrt(2), -3, np.pi, 9], [-1, -3, -5, -8], [np.sin(42), np.cos(24), 2.718, 99]])

print("기존 행렬의 대각합:", np.trace(D))
print("기저 변환 후 대각합:", np.trace(np.linalg.inv(T) @ D @ T))

기존 행렬의 대각합: 10
기저 변환 후 대각합: 10.000000000000004


In [2]:
def C(n, axis="z"):
    th = 2 * np.pi / n
    c, s = np.cos(th), np.sin(th)
    if axis == "z":
        return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])
    if axis == "y":
        return np.array([[c, 0, s], [0, 1.0, 0], [-s, 0, c]])
    return np.array([[1.0, 0, 0], [0, c, -s], [0, s, c]])


def sigma_v(d):
    th = np.radians(d)
    return np.array([[np.cos(2 * th), np.sin(2 * th), 0], [np.sin(2 * th), -np.cos(2 * th), 0], [0, 0, 1.0]])


def orbital_rep(op, coords, labels):
    """s 오비탈 기저에 대한 표현 행렬"""
    n = len(coords)
    M = np.zeros((n, n))
    moved = coords @ op.T
    for i, (p, l) in enumerate(zip(moved, labels)):
        for j, (q, m) in enumerate(zip(coords, labels)):
            if l == m and np.allclose(p, q, atol=1e-6):
                M[j, i] = 1
                break
    return M


r, z = 0.9375, -0.3810
ammonia = np.array(
    [
        [0, 0, 0],
        [r, 0, z],
        [r * np.cos(2 * np.pi / 3), r * np.sin(2 * np.pi / 3), z],
        [r * np.cos(4 * np.pi / 3), r * np.sin(4 * np.pi / 3), z],
    ]
)
labels = ["N", "H", "H", "H"]

ops = {
    "E": np.eye(3),
    "C3": C(3),
    "C3^2": C(3) @ C(3),
    "σ_a": sigma_v(0),
    "σ_b": sigma_v(120),
    "σ_c": sigma_v(240),
}
orb_reps = {k: orbital_rep(v, ammonia, labels) for k, v in ops.items()}

new_basis = [
    ("sN", np.array([1, 0, 0, 0])),
    ("sA + sB + sC", np.array([0, 1 / np.sqrt(3), 1 / np.sqrt(3), 1 / np.sqrt(3)])),
    ("2sA - sB - sC", np.array([0, 2 / np.sqrt(6), -1 / np.sqrt(6), -1 / np.sqrt(6)])),
    ("sB - sC", np.array([0, 0, 1 / np.sqrt(2), -1 / np.sqrt(2)])),
]

new_basis_names = [b[0] for b in new_basis]
T = np.column_stack([b[1] for b in new_basis])

blocks = {"s_N": slice(0, 1), "sA + sB + sC": slice(1, 2), "2x2": slice(2, 4)}

print(" " * 12 + "".join(f"{k:>7}" for k in ops))
for name, sl in blocks.items():
    row = [round(np.trace((T.T @ D @ T)[sl, sl])) for D in orb_reps.values()]
    print(f"{name:>12}" + "".join(f"{v:>7d}" for v in row))
print(f"{'total':>12}" + "".join(f"{round(np.trace(D)):>7d}" for D in orb_reps.values()))

                  E     C3   C3^2    σ_a    σ_b    σ_c
         s_N      1      1      1      1      1      1
sA + sB + sC      1      1      1      1      1      1
         2x2      2     -1     -1      0      0      0
       total      4      1      1      2      2      2


In [3]:
c3v = {"A1": [1, 1, 1], "A2": [1, 1, -1], "E": [2, -1, 0]}  # 지표표
sizes = [1, 2, 3]  # 각 유형에 속하는 조작의 개수
h = sum(sizes)  # 군의 위수


def reduce(chi, table, sizes):
    return {name: round(sum(g * a * b for g, a, b in zip(sizes, chi, row)) / sum(sizes)) for name, row in table.items()}


chi_nh3 = [4, 1, 2]
print("암모니아 분자의 s 오비탈 표현의 지표:", chi_nh3)
print()

for name, row in c3v.items():
    terms = " + ".join(f"{g} * {a} * {b}" for g, a, b in zip(sizes, chi_nh3, row))
    n = sum(g * a * b for g, a, b in zip(sizes, chi_nh3, row)) / h
    print(f"n({name}) = (1/{h})[{terms}] = {n:.0f}")
print()
print("축약 결과:", reduce(chi_nh3, c3v, sizes))

암모니아 분자의 s 오비탈 표현의 지표: [4, 1, 2]

n(A1) = (1/6)[1 * 4 * 1 + 2 * 1 * 1 + 3 * 2 * 1] = 2
n(A2) = (1/6)[1 * 4 * 1 + 2 * 1 * 1 + 3 * 2 * -1] = 0
n(E) = (1/6)[1 * 4 * 2 + 2 * 1 * -1 + 3 * 2 * 0] = 1

축약 결과: {'A1': 2, 'A2': 0, 'E': 1}


In [4]:
def sigma(plane):
    m = np.eye(3)
    m[{"yz": 0, "xz": 1, "xy": 2}[plane]] *= -1
    return m


r, theta = 0.9572, np.radians(104.52)
water = np.array(
    [
        [0, 0, 0],
        [0, r * np.sin(theta / 2), -r * np.cos(theta / 2)],
        [0, -r * np.sin(theta / 2), -r * np.cos(theta / 2)],
    ]
)
labels = ["O", "H", "H"]

ops_water = {"E": np.eye(3), "C2(z)": C(2), "σ(xz)": sigma("xz"), "σ(yz)": sigma("yz")}

chi_3n = []
print(" " * 8 + f"{'제자리 원자':>8}{'tr(R)':>8}{'χ':>8}")
for k, op in ops_water.items():
    n_fixed = sum(np.allclose(p, q, atol=1e-6) for p, q in zip(water @ op.T, water))
    chi = n_fixed * np.trace(op)
    chi_3n.append(round(chi))
    print(f"{k:>8}{n_fixed:>12}{np.trace(op):>8.0f}{chi:>8.0f}")

print()
print("χ(3N) =", chi_3n)

          제자리 원자   tr(R)       χ
       E           3       3       9
   C2(z)           1      -1      -1
   σ(xz)           1       1       1
   σ(yz)           3       1       3

χ(3N) = [9, -1, 1, 3]


In [5]:
c2v = {"A1": [1, 1, 1, 1], "A2": [1, 1, -1, -1], "B1": [1, -1, 1, -1], "B2": [1, -1, -1, 1]}
sizes_water = [1, 1, 1, 1]

print("축약 결과:", reduce(chi_3n, c2v, sizes_water))

축약 결과: {'A1': 3, 'A2': 1, 'B1': 2, 'B2': 3}
